# Layer 2 evaluation

Run `python -m sim.run_sim` and `python -m ml.train` first. This notebook re-loads the
event log, rebuilds features with the shared pipeline, and renders the charts for the
submission: PR curve, Precision@k vs alert budget, SHAP summary, and the lift table
against the Layer-1 threshold-only baseline.

Method and metric choices are grounded in Dal Pozzolo et al. 2018 (IEEE TNNLS),
Dal Pozzolo et al. 2014 (ESWA), Davis & Goadrich 2006 (ICML), Whitrow et al. 2009,
Bahnsen et al. 2016, and Carcillo et al. 2021 (Information Sciences).

In [ ]:
import json, pathlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
import lightgbm as lgb, joblib, shap
from ml import features, train

HERE = pathlib.Path('.').resolve()
events = pd.read_parquet('events.parquet')
feats = features.compute_features_batch(events)
tr, va, te = train.time_split(feats)
fcols = features.feature_columns(feats)
print(len(events), 'events;', len(feats), 'feature rows;', f"{(feats.label=='attack').mean():.1%} attack")

In [ ]:
# load the trained artifacts
booster = lgb.Booster(model_file='ml/model.txt')
iforest = joblib.load('ml/iforest.pkl')
calib = joblib.load('ml/calibrator.pkl')
flist = json.loads(pathlib.Path('ml/feature_list.json').read_text())
cutoff = json.loads(pathlib.Path('ml/threshold.json').read_text())['cutoff']
base = [f for f in flist if f != 'anomaly_score']

te = te.copy()
te['anomaly_score'] = -iforest.score_samples(te[base].to_numpy())
raw = booster.predict(te[flist].to_numpy())
te['score'] = calib.predict_proba(raw.reshape(-1,1))[:,1]
y = (te.label=='attack').astype(int)
print('test AUC-PR:', round(average_precision_score(y, te.score), 3))

In [ ]:
# PR curve (Davis & Goadrich 2006: the right curve under heavy imbalance)
p, r, _ = precision_recall_curve(y, te.score)
plt.figure(figsize=(5,4))
plt.plot(r, p); plt.xlabel('recall'); plt.ylabel('precision')
plt.title(f'Layer 2 PR curve (AP={average_precision_score(y, te.score):.3f})')
plt.grid(alpha=.3); plt.show()

In [ ]:
# Precision@k vs the analyst alert budget (Dal Pozzolo 2014)
ks = [2,4,6,8,12,20]
rows = [(k,)+train.precision_recall_at_k(te, 'score', k)[:2] for k in ks]
bud = pd.DataFrame(rows, columns=['k_per_hour','precision','recall'])
bud

In [ ]:
# SHAP: why the model fires
expl = shap.TreeExplainer(booster)
sv = expl.shap_values(te[flist])
shap.summary_plot(sv, te[flist], show=True)

In [ ]:
# Lift vs Layer 1 (threshold-5 only) on the same held-out time slice
base_stats = train.layer1_baseline(events, (te.ts.min(), te.ts.max()))
model_caught = int((te[te.label=='attack'].groupby('campaign_id').score.max() >= cutoff).sum())
print('attack campaigns in test slice :', base_stats['campaigns'])
print('Layer 1 (threshold) caught     :', base_stats['caught'],
      f"(median {base_stats['median_guesses_before_catch']:.0f} guesses before catch)")
print('Layer 2 (this model) caught    :', model_caught)